# SafeAlert Runner

Interactive wrapper around `scripts/run_pilot.py`.

Set `MODEL` and `RUN_TYPE` in Cell 2, then run all cells.

Results are appended to `results/raw/`. The same run ID on the same date across the public and private datasets merges into one JSONL file.

In [ ]:
# Configuration
import os
import sys
from pathlib import Path

sys.path.insert(0, "..")

if Path.cwd().name == "notebooks":
    os.chdir(Path("..").resolve())

from scripts.run_pilot import build_run_id, run_pilot

MODEL = "gpt4o"        # "gpt4o" or "llama"
RUN_TYPE = "pre_remediation"  # "pre_remediation" or "post_remediation"
PUBLIC_DATASET = "dataset/public/safealert_dataset_v1_public.csv"
PRIVATE_DATASET = "dataset/private/safealert_dataset_v1_private.csv"

In [ ]:
# Dry-run validation
try:
    run_pilot(PUBLIC_DATASET, MODEL, RUN_TYPE, dry_run=True)
    run_pilot(PRIVATE_DATASET, MODEL, RUN_TYPE, dry_run=True)
    print("Validation passed. Ready to run.")
except Exception as exc:
    print(f"Validation failed: {exc}")
    raise SystemExit

In [ ]:
# Run public dataset
from datetime import datetime

print(f"[{datetime.now().isoformat(timespec='seconds')}] Starting public dataset run.")
run_pilot(PUBLIC_DATASET, MODEL, RUN_TYPE)
print(f"[{datetime.now().isoformat(timespec='seconds')}] Finished public dataset run.")

In [ ]:
# Run private dataset
from datetime import datetime

print(f"[{datetime.now().isoformat(timespec='seconds')}] Starting private dataset run.")
run_pilot(PRIVATE_DATASET, MODEL, RUN_TYPE)
print(f"[{datetime.now().isoformat(timespec='seconds')}] Finished private dataset run.")

In [ ]:
# Post-run check
from pathlib import Path

run_id = build_run_id(MODEL, RUN_TYPE)
output_path = Path("results/raw") / f"{run_id}.jsonl"

if output_path.exists():
    with output_path.open("r", encoding="utf-8") as jsonl_file:
        line_count = sum(1 for _ in jsonl_file)
    print(f"Output file: {output_path}")
    print(f"Line count: {line_count}")
    if line_count != 310:
        print("WARNING: line count is not 310.")
else:
    print(f"WARNING: output file not found: {output_path}")